<a href="https://colab.research.google.com/github/Harikeshkaant/DML/blob/main/Final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1)Import libraires

In [11]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, log_loss
import statsmodels.api as sm

np.random.seed(42)


# 2. SYNTHETIC DATA GENERATION

In [3]:
def generate_data(n=2000, p=10):
    """
    Generate synthetic data with:
    - Nonlinear outcome function
    - Nonlinear propensity score
    - Heterogeneous treatment effect (CATE)
    """

    X = np.random.normal(0, 1, (n, p))

    # True heterogeneous treatment effect
    true_theta = 2 + X[:, 0] + 0.5 * (X[:, 1] ** 2)

    # Propensity score (nonlinear)
    logits = X[:, 0] - 0.5 * X[:, 1] + 0.3 * (X[:, 2] ** 2)
    prob = 1 / (1 + np.exp(-logits))
    T = np.random.binomial(1, prob)

    # Baseline outcome
    g_X = X[:, 0]**2 + np.sin(X[:, 1]) + 0.5 * X[:, 2]

    # Final outcome
    Y = true_theta * T + g_X + np.random.normal(0, 1, n)

    return X, T, Y, true_theta


# Generate data
X, T, Y, true_theta = generate_data()

# 3. DML WITH CROSS-FITTING (K=5)

In [4]:
K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=42)

Y_residuals = np.zeros(len(Y))
T_residuals = np.zeros(len(T))

# Store predictions for evaluation
m_hat_full = np.zeros(len(Y))
e_hat_full = np.zeros(len(T))

for train_idx, test_idx in kf.split(X):

    X_train, X_test = X[train_idx], X[test_idx]
    Y_train, Y_test = Y[train_idx], Y[test_idx]
    T_train, T_test = T[train_idx], T[test_idx]


    # Stage 1: Nuisance Models


    # Outcome model
    model_y = Lasso(alpha=0.01)
    model_y.fit(X_train, Y_train)
    m_hat = model_y.predict(X_test)

    # Propensity model
    model_t = LogisticRegression(max_iter=1000)
    model_t.fit(X_train, T_train)
    e_hat = model_t.predict_proba(X_test)[:, 1]

    # Store predictions
    m_hat_full[test_idx] = m_hat
    e_hat_full[test_idx] = e_hat


    # Stage 2: Residuals (Orthogonalization)


    Y_residuals[test_idx] = Y_test - m_hat
    T_residuals[test_idx] = T_test - e_hat





# 3. ATE ESTIMATION

In [ ]:
X_final = sm.add_constant(T_residuals)
model_final = sm.OLS(Y_residuals, X_final).fit()

tau_hat = model_final.params[1]
std_error = model_final.bse[1]
p_value = model_final.pvalues[1]

print("===== ATE ESTIMATION =====")
print(f"Estimated ATE (tau): {tau_hat:.4f}")
print(f"Standard Error: {std_error:.4f}")
print(f"P-value: {p_value:.6f}")

===== ATE ESTIMATION =====
Estimated ATE (tau): 2.3463
Standard Error: 0.1089
P-value: 0.0000


# 4. CATE ESTIMATION (METHOD 1: INTERACTION-BASED DML)

In [5]:
# Interaction terms
TX = T_residuals.reshape(-1, 1) * X

X_cate = np.column_stack((T_residuals, TX))
X_cate = sm.add_constant(X_cate)

model_cate = sm.OLS(Y_residuals, X_cate).fit()

beta = model_cate.params

# Estimated CATE
theta_hat_interaction = beta[1] + TX @ beta[2:]



# 6. EVALUATION

In [9]:
def evaluate(true, pred, name):
    bias = np.mean(pred - true)
    rmse = np.sqrt(mean_squared_error(true, pred))

    print(f"\n===== {name} =====")
    print(f"Bias: {bias:.4f}")
    print(f"RMSE: {rmse:.4f}")

# Evaluate both methods
evaluate(true_theta, theta_hat_interaction, "CATE (Interaction DML)")
evaluate(true_theta, theta_hat_rlearner, "CATE (R-Learner)")




===== CATE (Interaction DML) =====
Bias: -0.1440
RMSE: 1.3522

===== CATE (R-Learner) =====
Bias: -0.1171
RMSE: 0.7745


# 7. NUISANCE MODEL PERFORMANCE


In [12]:

y_mse = mean_squared_error(Y, m_hat_full)
t_logloss = log_loss(T, e_hat_full)

print("\n===== NUISANCE MODEL PERFORMANCE =====")
print(f"Outcome Model MSE: {y_mse:.4f}")
print(f"Propensity Model LogLoss: {t_logloss:.4f}")


===== NUISANCE MODEL PERFORMANCE =====
Outcome Model MSE: 5.7683
Propensity Model LogLoss: 0.5781


# 8. SAVE RESULTS

In [13]:
results_df = pd.DataFrame({
    "True_CATE": true_theta,
    "CATE_Interaction": theta_hat_interaction,
    "CATE_RLearner": theta_hat_rlearner
})

results_df.to_csv("cate_results_improved.csv", index=False)

print("\nResults saved to cate_results_improved.csv")


Results saved to cate_results_improved.csv


# Summary of my project


1. ATE is estimated using orthogonalized residuals, ensuring unbiased estimation.

2. Interaction-based DML improves over naive models by allowing treatment
   effects to vary with covariates, but may still suffer if the model is linear.

3. R-Learner provides a more flexible and robust way to estimate CATE,
   especially in high-dimensional and nonlinear settings.

4. Bias and RMSE help evaluate how well estimated CATE matches the true effect.

5. Nuisance model performance is critical — poor estimation of propensity
   scores or outcome functions directly impacts causal estimates.